In [1]:
#using Pkg
#Pkg.add("JSON")
#Pkg.add("DotEnv")
#Pkg.add("HTTP")
#Pkg.add("JSON")
#Pkg.add("URIs")
#Pkg.add("ConfigEnv")

using DotEnv, HTTP, URIs,JSON


This is a modified version of the get_data program in the api documentation.  Designed here to illustrate error correction methods

In [2]:
function get_data(property) 
    cfg = DotEnv.config()
    #a bogus token which works for public request
    default_token="eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiJ9.eyJpZCI6IjhjNzRhMjY0YjE0Yj"
    if isempty(cfg)
        response = Dict(
            "error" => "configuration",
            "error_description" => ".env files doesn't exist or is empty"
            )
        return JSON.json(response)
    elseif cfg["api_url"] == ""
        response = Dict(
            "error" => "api_url",
            "error_description" => "there is no api_url, can't return output"
            )
        return JSON.json(response)
    elseif cfg["access_token"] == ""
        #assuming this is a public request
        cfg["access_token"] = default_token
    end
                
    api_url = string(cfg["api_url"],"/",property)
    new_headers = Dict(
        "Accept" => "application/json",
        "Authorization" => string("Bearer ",cfg["access_token"]),
    )
    response = HTTP.request("GET", api_url, new_headers)
        
    r = String(response.body)
    check = JSON.parse(r)
    if "error" in keys(check)
        println("detected error")
    end
    return r
end

get_data (generic function with 1 method)

In [10]:

placements = get_data("from_metadata");


In [12]:
md = JSON.parse(placements);

In [13]:
n = 0
for m in md
    if n < 1
        println("........")
        for(key, value) in m
            println(key, " => ", value)
        end
        println("........")
    end
    n += 1
end


........
enrolldate => 2022-11-16 03:55:06
degreeendyear => 2023
fname => Wan
state => nothing
curinst => University of California Los Angeles (UCLA)
name => Econometrics
lname => Zhang
category_id => 2
degreeinst_oid => 36
from_shortname => Economics, UCLA
from_oid => 36
addr1 => nothing
country => United States
rank => 20
country_code => US
degreeinst => University of California Los Angeles (UCLA)
position_type_id => 20
degreetype => PhD
addr2 => nothing
degreebegyear => 2016
ethnic => 
curpos => ds
aid => 59156
latitude => 34.068921
from_institution_id => 6
gender => Male
city => nothing
longitude => -118.4451811
from_department => Economics
from_institution_name => University of California Los Angeles (UCLA)
categories => 7
race => Asian
degree_id => 13
........


To flag the entries I use an elementary method.  First I'll create an empty set to store the placements I have already checked called `records` and another set to hold the aids I want to flag called `flags`.

Then I'll iterate through the placements one by one and check whether each placement is already in `records`.  If it is there already I'll try to add the aid to `flags`.  If it isn't there I'll add the record to `records`

In [6]:
records = Set()
flags = Set()
#x = [1,2,3]
#push!(s,[1,2,3])
#println(s)

Set{Any}()

In [7]:
for placement in placements
    t = [placement["aid"],placement["to_oid"],placement["from_oid"],placement["year"],placement["postype"]]
    #println( "checking ", t)
    if t in records
        push!(flags,placement["aid"])
    else
        push!(records,t)
    end 
end 
        

LoadError: MethodError: no method matching getindex(::Char, ::String)
[0mClosest candidates are:
[0m  getindex(::AbstractChar) at ~/julia/julia-1.7.3/share/julia/base/char.jl:202
[0m  getindex(::AbstractChar, [91m::Integer[39m) at ~/julia/julia-1.7.3/share/julia/base/char.jl:203
[0m  getindex(::AbstractChar, [91m::Integer...[39m) at ~/julia/julia-1.7.3/share/julia/base/char.jl:204
[0m  ...

In [8]:
println(length(records))
println(length(flags))

0
0


In [9]:
println(flags)

Set{Any}()
